# Model 7: Nuisance Evidence Retrieval

Paired open-set test of Dalvin Model 3d versus a frozen nuisance transformation for raw similarity evidence. These are local validation scores, not Kaggle scores. A submission is generated only when the predeclared validation gate passes.

## Source snapshot
CPU is used by SIFT matching and the scikit-learn/LightGBM models. Initial mutual SIFT verification is expensive; cached pairs are reused on reruns.

In [ ]:
from pathlib import Path
import json, subprocess, sys
CANDIDATES=[Path.cwd(), Path.cwd()/"Models"/"Abhay"/"Model7", Path.cwd()/"Model7"]
ROOT=next((p for p in CANDIDATES if (p/"paired_eval.py").exists()),Path.cwd())
RUN_DIR=ROOT / "local_runs" / "model7_paired_2052"
SOURCE=RUN_DIR / "source"
SOURCE.mkdir(parents=True,exist_ok=True)
(SOURCE / 'paired_eval.py').write_text('"""Paired open-set comparison: Dalvin 3d features versus nuisance raw evidence."""\nimport argparse\nimport hashlib\nimport json\nimport time\nimport warnings\nfrom pathlib import Path\n\nimport cv2\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom sklearn.utils.extmath import randomized_svd\nfrom threadpoolctl import threadpool_limits\n\nimport dalvin_core as b\nimport data_protocol as protocol\n\n\ndef nuisance_features(indices, dimensions=16, strength=.5, seed=2052):\n    """Every non-singleton fitting identity contributes equal covariance weight."""\n    transformed = b.X.copy()\n    directions = {}\n    for species in (\'hyena\', \'leopard\'):\n        fit = indices[b.SP[indices] == species]\n        residuals = []\n        for identity in np.unique(b.Y[fit]):\n            images = b.X[fit[b.Y[fit] == identity]]\n            if len(images) > 1:\n                residuals.append((images - images.mean(axis=0)) / np.sqrt(len(images) - 1))\n        if not residuals:\n            raise ValueError(\'No repeated fitting identities for nuisance estimation\')\n        residuals = np.vstack(residuals)\n        # Match the research implementation: estimate 64 components, use the first 16.\n        _, _, u = randomized_svd(residuals, n_components=min(64, min(residuals.shape) - 1),\n                                 n_iter=4, random_state=seed)\n        u = u[:dimensions]\n        directions[species] = u\n        ix = np.flatnonzero(b.SP == species)\n        transformed[ix] = b.unit(b.X[ix] - strength * (b.X[ix] @ u.T) @ u)\n    return transformed, directions\n\n\ndef stats(frame):\n    report = {}\n    for method in (\'baseline\', \'nuisance\'):\n        col = method + \'_rr5\'\n        report[method] = {\'map5\': float(frame[col].mean()),\n                          \'known_map5\': float(frame.loc[~frame.unknown, col].mean()),\n                          \'unknown_map5\': float(frame.loc[frame.unknown, col].mean()),\n                          \'top1\': float((frame[col] == 1).mean())}\n    report[\'delta\'] = report[\'nuisance\'][\'map5\'] - report[\'baseline\'][\'map5\']\n    return report\n\n\ndef main():\n    ap = argparse.ArgumentParser(description=__doc__)\n    ap.add_argument(\'--data\', type=Path, default=Path.home() / \'Downloads/reid_student_data_main\')\n    ap.add_argument(\'--output\', type=Path, default=Path(\'local_runs/paired_2052\'))\n    ap.add_argument(\'--seed\', type=int, default=2052)\n    ap.add_argument(\'--sims\', type=int, default=12)\n    ap.add_argument(\'--workers\', type=int, default=4)\n    ap.add_argument(\'--neighbours\', type=int, default=150)\n    ap.add_argument(\'--trees\', type=int, default=400)\n    ap.add_argument(\'--predict-test\', action=\'store_true\')\n    args = ap.parse_args()\n    if min(args.sims, args.workers, args.neighbours, args.trees) < 1:\n        ap.error(\'Counts must be positive\')\n    args.output.mkdir(parents=True, exist_ok=True)\n    config = {k: str(v) if isinstance(v, Path) else v for k, v in vars(args).items()}\n    config.update(rounds=3, nuisance_dimensions=16, nuisance_strength=.5,\n                  evaluation_episodes=[args.seed + 20000 + j for j in range(3)],\n                  gate=\'positive pooled and each-episode delta; known and unknown MAP5 do not fall\')\n    (args.output / \'config.json\').write_text(json.dumps(config, indent=2))\n    cv2.setNumThreads(1)\n    b.PARAMS.update(n_estimators=args.trees, n_jobs=args.workers, random_state=args.seed)\n    warnings.filterwarnings(\'ignore\', message=\'Only one sample available.*\', category=UserWarning)\n    started = time.monotonic()\n    train, test, all_df = protocol.load_data(args.data)\n    roles = protocol.allocate_roles(train, 3, args.seed)\n    roles[\'metric_fit\'] = roles.pop(\'calibration\')\n    manifest = pd.DataFrame([{\'id\': b.IDS[i], \'species\': b.SP[i], \'identity\': b.Y[i], \'role\': name}\n                             for name, ix in roles.items() for i in ix])\n    assert manifest.groupby(\'identity\').role.nunique().max() == 1\n    manifest.to_csv(args.output / \'identity_roles.csv\', index=False)\n    nx, directions = nuisance_features(roles[\'metric_fit\'], seed=args.seed)\n    joblib.dump(directions, args.output / \'nuisance_directions.joblib\')\n    cache = protocol.SiftCache(args.data, args.output, args.workers, args.neighbours)\n    models = {\'baseline\': [], \'nuisance\': []}\n    splits = []\n\n    def split(ix, seed, name, link):\n        gi, qi, truth = protocol.split_episode(ix, seed, link=link)\n        truth_map = dict(zip(qi, truth))\n        for i in gi:\n            splits.append({\'episode\': seed, \'role\': name, \'id\': b.IDS[i], \'assignment\': \'gallery\'})\n        for i in qi:\n            splits.append({\'episode\': seed, \'role\': name, \'id\': b.IDS[i],\n                           \'assignment\': \'unknown_query\' if truth_map[i] == protocol.NEW else \'known_query\'})\n        return gi, qi, truth\n\n    def frames(gi, qi, truth):\n        result = {}\n        for method in models:\n            b.NX = b.X if method == \'baseline\' else nx\n            result[method] = protocol.frame(gi, qi, truth)\n        assert np.array_equal(result[\'baseline\'][\'order\'], result[\'nuisance\'][\'order\'])\n        assert result[\'baseline\'][\'c\'].ident.equals(result[\'nuisance\'][\'c\'].ident)\n        return result\n\n    for stage in range(3):\n        role = f\'stage{stage + 1}\'\n        cache.activate(roles[role])\n        training = {name: [] for name in models}\n        for j in range(args.sims):\n            gi, qi, truth = split(roles[role], args.seed + 1000 * (stage + 1) + j,\n                                 role, b.REGIMES[j % len(b.REGIMES)])\n            paired = frames(gi, qi, truth)\n            for method, frame in paired.items():\n                for model in models[method]:\n                    frame = b.augment(frame, b.predict(model, frame))\n                training[method].append(frame)\n            print(f\'{role} simulation {j + 1}/{args.sims}\', flush=True)\n        for method in models:\n            print(f\'Fitting {method} {role}\', flush=True)\n            models[method].append(b.fit(training[method]))\n        joblib.dump(models, args.output / \'paired_models.joblib\')\n    # Everything supervised is frozen before opening validation identities.\n    cache.activate(roles[\'validation\'])\n    results = []\n    for j in range(3):\n        seed = args.seed + 20000 + j\n        gi, qi, truth = split(roles[\'validation\'], seed, \'validation\', (.95, 111))\n        paired = frames(gi, qi, truth)\n        frame = pd.DataFrame({\'episode\': seed, \'id\': b.IDS[paired[\'baseline\'][\'order\']],\n                              \'identity\': b.Y[paired[\'baseline\'][\'order\']],\n                              \'species\': b.SP[paired[\'baseline\'][\'order\']],\n                              \'truth\': paired[\'baseline\'][\'truth\']})\n        frame[\'unknown\'] = frame.truth == protocol.NEW\n        for method, s in paired.items():\n            s, r = protocol.through_stack(s, models[method])\n            predictions = b.rank(s, r)\n            frame[method + \'_rr5\'] = [next((1 / (k + 1) for k, label in enumerate(row) if label == target), 0.)\n                                       for row, target in zip(predictions, frame.truth)]\n            frame[method + \'_predictions\'] = [\' \'.join(row) for row in predictions]\n        results.append(frame)\n        print(\'Validation\', seed, json.dumps(stats(frame)), flush=True)\n    result = pd.concat(results, ignore_index=True)\n    result.to_csv(args.output / \'validation_predictions.csv\', index=False)\n    pd.DataFrame(splits).to_csv(args.output / \'episode_splits.csv\', index=False)\n    report = stats(result)\n    report[\'episodes\'] = {str(seed): stats(part) for seed, part in result.groupby(\'episode\')}\n    report[\'species\'] = {str(sp): stats(part) for sp, part in result.groupby(\'species\')}\n    report[\'query_appearances\'] = len(result)\n    report[\'unique_queries\'] = result.id.nunique()\n    report[\'gate_passed\'] = bool(report[\'delta\'] > 0 and all(v[\'delta\'] > 0 for v in report[\'episodes\'].values())\n                                 and all(report[\'nuisance\'][k] >= report[\'baseline\'][k]\n                                         for k in (\'known_map5\', \'unknown_map5\')))\n    report[\'elapsed_seconds\'] = time.monotonic() - started\n    (args.output / \'metrics.json\').write_text(json.dumps(report, indent=2))\n    print(json.dumps(report, indent=2), flush=True)\n    if args.predict_test and report[\'gate_passed\']:\n        cache.activate(np.arange(len(all_df)))\n        gi, qi = np.arange(len(train)), np.arange(len(train), len(all_df))\n        b.NX = nx\n        s, r = protocol.through_stack(protocol.frame(gi, qi), models[\'nuisance\'])\n        predictions = b.rank(s, r)\n        sub = pd.DataFrame({\'id\': b.IDS[s[\'order\']], \'predictions\': [\' \'.join(row) for row in predictions]})\n        sample = pd.read_csv(args.data / \'sample_submission.csv\')\n        sub = sample[[\'id\']].merge(sub, on=\'id\', validate=\'one_to_one\')\n        allowed = train.groupby(\'species\').identity.apply(set).to_dict()\n        species = test.set_index(\'id\').species\n        assert len(sub) == len(test) and sub.id.is_unique and not sub.predictions.isna().any()\n        for row in sub.itertuples():\n            labels = row.predictions.split()\n            assert len(labels) == len(set(labels)) == 5\n            assert all(t == protocol.NEW or t in allowed[species[row.id]] for t in labels)\n        sub.to_csv(args.output / \'submission_nuisance.csv\', index=False)\n        print(\'Saved submission_nuisance.csv\', flush=True)\n    elif args.predict_test:\n        print(\'Validation gate failed; submission generation skipped.\', flush=True)\n\n\nif __name__ == \'__main__\':\n    with threadpool_limits(limits=4, user_api=\'blas\'):\n        main()\n',encoding="utf-8")
(SOURCE / 'data_protocol.py').write_text('"""Data loading, role-local mutual SIFT and identity-disjoint episode helpers."""\nimport os\nimport hashlib\nfrom pathlib import Path\nfrom concurrent.futures import ThreadPoolExecutor\nfrom functools import lru_cache\nfrom threading import Lock\nimport numpy as np\nimport pandas as pd\nimport cv2\nfrom scipy import sparse\nimport dalvin_core as b\nNEW=\'new_individual\'\ndef find_sift(root, split):\n    """Kaggle mounts SIFT beside the metadata folder, sometimes as extracted NPZ directories."""\n    name = f"sift_features_{split}.npz"\n    for parent in (root, root.parent, root.parent.parent):\n        candidate = parent / name\n        if candidate.exists():\n            return candidate\n    raise FileNotFoundError(f"Cannot find {name} in {root} or its two parent directories. "\n                            "Attach the full projectwild3 competition input, including both SIFT folders.")\n\n\nclass SiftSource:\n    def __init__(self, path):\n        self.archive = None\n        self.files = {}\n        if path.is_file():\n            self.archive = np.load(path)\n        else:\n            for file in path.rglob("*.npy"):\n                if file.stem in self.files:\n                    raise ValueError(f"Duplicate SIFT key {file.stem} in {path}")\n                self.files[file.stem] = file\n            if not self.files:\n                raise ValueError(f"No extracted .npy SIFT arrays in {path}")\n\n    def __getitem__(self, key):\n        return self.archive[key] if self.archive is not None else np.load(self.files[key], allow_pickle=False)\n\n\ndef load_data(root):\n    train = pd.read_csv(root / "train.csv")\n    test = pd.read_csv(root / "test.csv")\n    ids = pd.read_csv(root / "miewid_ids.csv")\n    all_df = pd.concat([train, test.assign(identity=None)], ignore_index=True)\n    if not all_df.id.is_unique or not ids.id.is_unique:\n        raise ValueError("Duplicate image IDs")\n    for col in ("species", "viewpoint", "identity"):\n        if col not in all_df:\n            raise ValueError(f"Missing column: {col}")\n    if train.identity.isna().any() or not set(all_df.species) <= {"hyena", "leopard"}:\n        raise ValueError("Invalid identity/species metadata")\n    raw = np.load(root / "miewid_features.npy", mmap_mode="r")\n    if len(raw) != len(ids) or set(ids.id) != set(all_df.id):\n        raise ValueError("Feature ID mapping does not match metadata")\n    ix = pd.Index(ids.id).get_indexer(all_df.id)\n    X = np.asarray(raw[ix], dtype=np.float32)\n    if not np.isfinite(X).all() or np.any(np.linalg.norm(X, axis=1) == 0):\n        raise ValueError("Features contain nonfinite values or zero vectors")\n    b.X = b.unit(X)\n    b.IDS = all_df.id.to_numpy()\n    b.Y = all_df.identity.to_numpy(dtype=object)\n    b.SP = all_df.species.to_numpy()\n    b.VIEW = all_df.viewpoint.map({v: i for i, v in enumerate(b.VIEWS)}).fillna(-1).to_numpy(int)\n    return train, test, all_df\n\n\nclass SiftCache:\n    """Role-local neighbour selection; thread workers share lazily loaded descriptors."""\n    def __init__(self, root, out, workers, neighbours):\n        paths = [find_sift(root, s) for s in ("train", "test")]\n        print("SIFT sources:", *paths, sep="\\n", flush=True)\n        self.z = [SiftSource(path) for path in paths]\n        self.read_lock = Lock()\n        self.workers, self.neighbours = workers, neighbours\n        fingerprint = hashlib.sha256(b.IDS.tobytes() if b.IDS.dtype != object else "\\n".join(b.IDS).encode())\n        fingerprint.update(np.ascontiguousarray(b.X).tobytes())\n        self.path = out / ("sift_mutual_" + fingerprint.hexdigest()[:16] + ".npz")\n        self.done = {}\n        if self.path.exists():\n            with np.load(self.path) as z:\n                self.done = {tuple(p): tuple(v) for p, v in zip(z["pairs"], z["values"])}\n\n    @lru_cache(maxsize=1200)\n    def descriptors(self, i):\n        iid = b.IDS[i]\n        z = self.z[0 if iid.startswith("tr_") else 1]\n        # NpzFile\'s shared ZIP handle is not safe for concurrent reads.\n        with self.read_lock:\n            d = np.asarray(z[f"{iid}_desc"], dtype=np.float32)\n            xy = np.asarray(z[f"{iid}_xy"], dtype=np.float32)\n        if d.ndim != 2 or xy.shape != (len(d), 2):\n            raise ValueError(f"Malformed SIFT arrays for {iid}")\n        d = np.sqrt(d / np.maximum(d.sum(1, keepdims=True), 1))\n        return d, xy\n\n    def verify(self, pair):\n        a, c = pair\n        da, xa = self.descriptors(a)\n        dc, xc = self.descriptors(c)\n        if min(len(da), len(dc)) < 4:\n            return 0, 0\n        bf = cv2.BFMatcher()\n        back = np.array([m.trainIdx for m in bf.match(dc, da)])\n        matches = bf.knnMatch(da, dc, k=2)\n        good = [m[0] for m in matches if len(m) == 2 and m[0].distance < .85*m[1].distance\n                and back[m[0].trainIdx] == m[0].queryIdx]\n        if len(good) < 4:\n            return len(good), 0\n        # Local RNG seed per task. OpenCV RNG state is thread-local.\n        cv2.setRNGSeed(int((int(a)*100003 + int(c)) % (2**31-1)))\n        _, mask = cv2.estimateAffinePartial2D(\n            np.float32([xa[m.queryIdx] for m in good]),\n            np.float32([xc[m.trainIdx] for m in good]),\n            method=cv2.RANSAC, ransacReprojThreshold=15.)\n        return len(good), int(mask.sum()) if mask is not None else 0\n\n    def activate(self, indices):\n        wanted = set()\n        for sp in ("hyena", "leopard"):\n            ix = indices[b.SP[indices] == sp]\n            # Chunked similarities avoid allocating the full role x role matrix.\n            for start in range(0, len(ix), 256):\n                q = ix[start:start+256]\n                s = b.X[q] @ b.X[ix].T\n                s[np.arange(len(q)), np.arange(start, start+len(q))] = -np.inf\n                k = min(self.neighbours, len(ix)-1)\n                nb = np.argsort(-s, axis=1)[:, :k]\n                for a, row in zip(q, nb):\n                    wanted.update((min(int(a), int(c)), max(int(a), int(c))) for c in ix[row])\n        missing = sorted(wanted - self.done.keys())\n        print(f"SIFT: {len(wanted)} role-local pairs; {len(missing)} uncached", flush=True)\n        with ThreadPoolExecutor(max_workers=self.workers) as pool:\n            for j, (pair, value) in enumerate(zip(missing, pool.map(self.verify, missing)), 1):\n                self.done[pair] = value\n                if j % 5000 == 0:\n                    self.save()\n                    print(f"  verified {j}/{len(missing)}", flush=True)\n        self.save()\n        # Only expose requested pairs, even when cache contains other roles/test.\n        pp = np.array(sorted(wanted), dtype=int).reshape(-1, 2)\n        vv = np.array([self.done[tuple(p)] for p in pp]).reshape(-1, 2)\n        n = len(b.X)\n        def matrix(v):\n            a, c = pp.T\n            return sparse.coo_matrix((np.r_[v, v], (np.r_[a, c], np.r_[c, a])), shape=(n, n)).tocsr()\n        b.GOOD, b.INL = matrix(vv[:, 0]), matrix(vv[:, 1])\n\n    def save(self):\n        """Persist completed verifications atomically so interrupted runs can resume."""\n        pairs = sorted(self.done)\n        tmp = self.path.with_suffix(".tmp.npz")\n        np.savez_compressed(tmp, pairs=np.array(pairs, dtype=np.int64).reshape(-1, 2),\n                            values=np.array([self.done[p] for p in pairs]).reshape(-1, 2))\n        os.replace(tmp, self.path)\n\n\ndef allocate_roles(train, rounds, seed):\n    rng = np.random.default_rng(seed)\n    roles = {f"stage{i+1}": [] for i in range(rounds)} | {"calibration": [], "validation": []}\n    for sp in ("hyena", "leopard"):\n        names = np.array(sorted(train.loc[train.species == sp, "identity"].unique()))\n        rng.shuffle(names)\n        n = max(1, int(round(.2*len(names))))\n        roles["validation"].extend(names[:n])\n        roles["calibration"].extend(names[n:2*n])\n        for i, block in enumerate(np.array_split(names[2*n:], rounds)):\n            roles[f"stage{i+1}"].extend(block)\n    sets = [set(v) for v in roles.values()]\n    assert all(not a & c for i, a in enumerate(sets) for c in sets[i+1:])\n    return {k: np.flatnonzero(train.identity.isin(v).to_numpy()) for k, v in roles.items()}\n\n\ndef split_episode(indices, seed, cap=32, link=(.95, 100)):\n    """Unknown identities absent from gallery; known-query duplicate groups kept whole."""\n    rng = np.random.default_rng(seed)\n    gallery, known, unknown = [], [], []\n    for sp in ("hyena", "leopard"):\n        ix = indices[b.SP[indices] == sp]\n        names = np.unique(b.Y[ix])\n        rng.shuffle(names)\n        nu = max(1, int(round(rng.uniform(.2, .35)*len(names))))\n        unseen = set(names[:nu])\n        for name in names:\n            rows = ix[b.Y[ix] == name].copy()\n            rng.shuffle(rows)\n            if name in unseen:\n                unknown.extend(rows[:cap])\n                continue\n            held = []\n            groups = b.duplicate_groups(rows, link)\n            target = min(max(1, int(round(.35*len(rows)))), cap)\n            for c in rng.permutation(np.unique(groups)):\n                members = rows[groups == c].tolist()\n                if len(held) < target and len(held)+len(members) < len(rows):\n                    held.extend(members)\n            # Never truncate a duplicate group after selecting it.\n            known.extend(held)\n            gallery.extend(i for i in rows if i not in set(held))\n    gi, qi = np.array(gallery, dtype=int), np.array(known+unknown, dtype=int)\n    if not len(known) or not len(unknown):\n        raise ValueError("Episode lacks known/unknown queries; use a larger role or different seed")\n    for sp in ("hyena", "leopard"):\n        if len(np.unique(b.Y[gi[b.SP[gi] == sp]])) < b.K or np.sum(b.SP[qi] == sp) < 6:\n            raise ValueError("Too few identities/queries per species; reduce --rounds or --candidates")\n    assert not set(b.Y[unknown]) & set(b.Y[gi])\n    assert set(b.Y[known]) <= set(b.Y[gi])\n    return gi, qi, np.where(np.isin(qi, unknown), NEW, b.Y[qi])\n\n\ndef frame(gi, qi, truth=None):\n    s = b.build_frames(gi, qi)\n    if truth is not None:\n        lookup = dict(zip(qi, truth))\n        s["truth"] = np.array([lookup[i] for i in s["order"]])\n        s["q"]["unk"] = (s["truth"] == NEW).astype(int)\n        s["c"]["y"] = (s["c"].ident.to_numpy() == s["truth"][s["c"].q.to_numpy()]).astype(int)\n    return s\n\n\ndef through_stack(s, models):\n    for i, m in enumerate(models):\n        r = b.predict(m, s)\n        if i < len(models)-1:\n            s = b.augment(s, r)\n    return s, r\n',encoding="utf-8")
(SOURCE / 'dalvin_core.py').write_text('"""Dalvin Model 3d functions; raw evidence can use a frozen nuisance projection."""\nimport os\nimport re\nimport cv2\nimport lightgbm as lgb\nimport numpy as np\nimport pandas as pd\nfrom multiprocessing import Pool\nfrom scipy import sparse\nfrom scipy.sparse.csgraph import connected_components\nfrom sklearn.decomposition import PCA\nfrom sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA\nK=15\nNEIGHBOURS=150\nMATCH_RATIO=.85\nRANSAC_PX=15.\nSIFT_PULL=.05\nVIEWS=[\'left\',\'right\',\'front\',\'back\',\'frontleft\',\'frontright\',\'backleft\',\'backright\',\'up\',\'down\']\nFLANK=np.array([0,1,2,2,0,1,0,1,2,2])\nREGIMES=[None,(.95,111),(.99,89),(.9,67)]\nPARAMS=dict(objective=\'binary\',learning_rate=.05,num_leaves=31,min_child_samples=30,subsample=.8,subsample_freq=1,colsample_bytree=.8,n_estimators=400,verbose=-1,random_state=42,n_jobs=4)\nINDEX_COLS=re.compile(r\'^(nb|nw|sb|sw)\\d$\')\n\n\n\n\n\n\n\ndef unit(a):\n    return a / np.linalg.norm(a, axis=1, keepdims=True)\n\n\ndef embed(gallery, labels, queries):\n    pca = PCA(min(384, gallery.shape[1], len(gallery) + len(queries) - 1), random_state=0).fit(np.r_[gallery, queries])      # unsupervised, sees the queries too\n    lda = LDA(solver="eigen", shrinkage=0.3).fit(pca.transform(gallery), labels)\n    project = lambda a: unit(lda.transform(pca.transform(a)))\n    return project(gallery), project(queries)\n\n\ndef per_identity(sim, inv, n):\n    """Per query: max and mean-of-top-2 similarity to each identity\'s gallery images."""\n    best, top2 = np.empty((sim.shape[0], n)), np.empty((sim.shape[0], n))\n    for j in range(n):\n        s = sim[:, inv == j]\n        best[:, j], top2[:, j] = s.max(1), np.sort(s, 1)[:, -2:].mean(1)\n    return best, top2\n\n\ndef species_features(gi, qi, sp):\n    """Query frame and (query, candidate) frame for one species; gi/qi are global image indices."""\n    Rg, Rq, yg, vg, vq = X[gi], X[qi], Y[gi], VIEW[gi], VIEW[qi]\n    Lg, Lq = embed(Rg, yg, Rq)\n    Rg, Rq = NX[gi], NX[qi]  # Frozen nuisance representation for raw evidence only.\n    ids, inv = np.unique(yg, return_inverse=True)\n    n, nq = len(ids), len(qi)\n    sim_l, sim_r = Lq @ Lg.T, Rq @ Rg.T\n    lda1, lda2 = per_identity(sim_l, inv, n)\n    raw1, raw2 = per_identity(sim_r, inv, n)\n    centroid = unit(np.stack([Lg[inv == j].mean(0) for j in range(n)]))\n    cen = Lq @ centroid.T\n    inl_g, good_g = INL[qi][:, gi].toarray().astype(np.float32), GOOD[qi][:, gi].toarray().astype(np.float32)\n    sift1, sift2 = per_identity(inl_g, inv, n)\n    good1, _ = per_identity(good_g, inv, n)\n    support = (inl_g >= 7).astype(np.float32) @ np.eye(n, dtype=np.float32)[inv]     # gallery images confirmed per identity\n    same_view = np.full((nq, n), np.nan)\n    for j in range(n):\n        for v in np.unique(vq):\n            cols = np.where((inv == j) & (vg == v))[0]\n            if len(cols):\n                same_view[vq == v, j] = sim_l[np.ix_(vq == v, cols)].max(1)\n    # viewpoint coverage: flanks of one animal look different, so unseen sides cannot be matched\n    by_view = np.stack([np.bincount(inv[vg == v], minlength=n) for v in range(len(VIEWS))], 1)\n    by_flank = np.stack([by_view[:, FLANK == f].sum(1) for f in range(3)], 1)\n    seen = vq >= 0\n    n_view = np.where(seen[:, None], by_view[:, vq.clip(0)].T, np.nan)\n    n_flank = np.where(seen[:, None], by_flank[:, FLANK[vq.clip(0)]].T, np.nan)\n    # transductive: neighbours among the queries themselves\n    qq_r, qq_l, qq_s = Rq @ Rq.T, Lq @ Lq.T, INL[qi][:, qi].toarray().astype(np.float32)\n    np.fill_diagonal(qq_r, -1), np.fill_diagonal(qq_l, -1)\n    nb, sb = np.argsort(-qq_l, 1)[:, :5], np.argsort(-qq_s, 1)[:, :5]\n    nbs, sbs = np.take_along_axis(qq_l, nb, 1), np.take_along_axis(qq_s, sb, 1)\n    t_l = np.take_along_axis(lda1, np.argsort(-lda1, 1)[:, :3], 1)\n    t_r = -np.sort(-raw1, 1)[:, :3]\n    t_s = -np.sort(-sift1, 1)[:, :3]\n    g_top = -np.sort(-sim_l, 1)\n    q = pd.DataFrame({\n        "sp": int(sp == "hyena"), "view": vq, "ng": len(gi),\n        "l1": t_l[:, 0], "l2": t_l[:, 1], "l3": t_l[:, 2], "r1": t_r[:, 0], "r2": t_r[:, 1], "r3": t_r[:, 2],\n        "lm12": t_l[:, 0] - t_l[:, 1], "rm12": t_r[:, 0] - t_r[:, 1],\n        "gd5": g_top[:, :5].mean(1), "gd20": g_top[:, :20].mean(1),\n        "qq1": nbs[:, 0], "qq3": nbs[:, :3].mean(1), "qq5": nbs.mean(1), "qqr1": qq_r.max(1),\n        "qn6": (qq_r > .6).sum(1), "qn7": (qq_r > .7).sum(1), "qn8": (qq_r > .8).sum(1),\n        "qq_minus_g": nbs[:, 0] - t_l[:, 0], "nb_g1": t_l[nb[:, 0], 0],\n        "s1": t_s[:, 0], "s2": t_s[:, 1], "s3": t_s[:, 2], "sg": good_g.max(1),\n        "qs1": sbs[:, 0], "qs3": sbs[:, :3].mean(1), "qsn10": (qq_s >= 10).sum(1), "qsn20": (qq_s >= 20).sum(1),\n        "qs_minus_s": sbs[:, 0] - t_s[:, 0], "nbs_s1": t_s[sb[:, 0], 0],\n    })\n    for k in range(5):\n        q[f"nb{k}"], q[f"nw{k}"], q[f"sb{k}"], q[f"sw{k}"] = nb[:, k], nbs[:, k], sb[:, k], sbs[:, k]\n    # candidates: best identities by embedding similarity, lightly boosted by SIFT evidence\n    cand = np.argsort(-(lda1 + SIFT_PULL * np.tanh(sift1 / 15)), 1)[:, :K]\n    r, c = np.repeat(np.arange(nq), K), cand.ravel()\n    nb_ev = np.max(np.minimum(nbs[:, :, None], np.stack([lda1[nb[:, k]] for k in range(5)], 1)), 1)\n    sb_ev = np.max(np.minimum(sbs[:, :, None], np.stack([sift1[sb[:, k]] for k in range(5)], 1)), 1)\n    rank_of = lambda m: (-m).argsort(1).argsort(1)[r, c]\n    cd = pd.DataFrame({\n        "q": r, "c": c, "rank": np.tile(np.arange(K), nq), "ident": ids[c],\n        "mxr": raw1[r, c], "m2r": raw2[r, c], "mxl": lda1[r, c], "m2l": lda2[r, c], "cl": cen[r, c],\n        "gapl": lda1[r, c] - t_l[r, 0], "gapr": raw1[r, c] - t_r[r, 0], "same": same_view[r, c],\n        "cnt": np.bincount(inv)[c], "nbev": nb_ev[r, c], "sbev": sb_ev[r, c],\n        "rk_r": rank_of(raw1), "rk_c": rank_of(cen), "rk_s": rank_of(sift1),\n        "n_view": n_view[r, c], "n_flank": n_flank[r, c], "support": support[r, c], "sx": sift1[r, c], "s2x": sift2[r, c], "gx": good1[r, c], "sgap": sift1[r, c] - t_s[r, 0],\n    })\n    return q, cd\n\n\ndef build_frames(gi, qi):\n    qs, cs, order, off = [], [], [], 0\n    for sp in ("hyena", "leopard"):\n        g, qq = gi[SP[gi] == sp], qi[SP[qi] == sp]\n        q, c = species_features(g, qq, sp)\n        c["q"] += off\n        q[[f"{p}{k}" for p in ("nb", "sb") for k in range(5)]] += off\n        qs.append(q), cs.append(c), order.append(qq)\n        off += len(qq)\n    return dict(q=pd.concat(qs, ignore_index=True), c=pd.concat(cs, ignore_index=True), order=np.concatenate(order))\n\n\ndef duplicate_groups(ix, link):\n    """Group images of one identity that are near-duplicates (same encounter); each image alone if link is None."""\n    if link is None:\n        return np.arange(len(ix))\n    edges = (X[ix] @ X[ix].T > link[0]) | (INL[ix][:, ix].toarray() >= link[1])\n    return connected_components(sparse.csr_matrix(edges))[1]\n\n\ndef columns(s):\n    qf = [c for c in s["q"].columns if c != "unk" and not INDEX_COLS.match(c)]\n    cf = [c for c in s["c"].columns if c not in ("q", "c", "ident", "y")]\n    return qf, cf\n\n\ndef candidate_matrix(s, qf, cf):\n    return pd.concat([s["c"][cf].reset_index(drop=True), s["q"].iloc[s["c"].q.to_numpy()][qf].reset_index(drop=True)], axis=1)\n\n\ndef fit(sims):\n    """new-individual detector, closed-set candidate scorer, and all-query candidate scorer."""\n    qf, cf = columns(sims[0])\n    known = lambda s: s["q"].unk.to_numpy()[s["c"].q.to_numpy()] == 0\n    cx = pd.concat([candidate_matrix(s, qf, cf) for s in sims])\n    cy = np.concatenate([s["c"].y.to_numpy() for s in sims])\n    ck = np.concatenate([known(s) for s in sims])\n    detector = lgb.LGBMClassifier(**{**PARAMS, "n_estimators": 300, "num_leaves": 15}).fit(\n        pd.concat([s["q"][qf] for s in sims]), np.concatenate([s["q"].unk.to_numpy() for s in sims]))\n    return detector, lgb.LGBMClassifier(**PARAMS).fit(cx[ck], cy[ck]), lgb.LGBMClassifier(**PARAMS).fit(cx, cy), (qf, cf)\n\n\ndef predict(models, s):\n    detector, closed, anyq, (qf, cf) = models\n    nq, cx = len(s["q"]), candidate_matrix(s, qf, cf)\n    p_new = detector.predict_proba(s["q"][qf])[:, 1]\n    raw = anyq.predict_proba(cx)[:, 1].reshape(nq, K)\n    within = closed.predict_proba(cx)[:, 1].reshape(nq, K)\n    within /= np.maximum(within.sum(1, keepdims=True), 1e-9)\n    return dict(pn=p_new, raw=raw, pc=within, pb=(p_new + np.clip(1 - raw.sum(1), 0, 1)) / 2)\n\n\ndef rank(s, r):\n    """Top-5 labels: P(new) vs (1 - P(new)) * P(identity | known)."""\n    nq = len(s["q"])\n    labels = np.concatenate([s["c"].ident.to_numpy(dtype=object).reshape(nq, K), np.full((nq, 1), "new_individual", dtype=object)], 1)\n    scores = np.concatenate([(1 - r["pb"])[:, None] * r["pc"], r["pb"][:, None]], 1)\n    return np.take_along_axis(labels, np.argsort(-scores, 1)[:, :5], 1)\n\n\ndef map5(preds, truth):\n    return np.mean([next((1 / (i + 1) for i, p in enumerate(row) if p == t), 0.0) for row, t in zip(preds, truth)])\n\n\ndef augment(s, r):\n    """Add the previous round\'s predictions for each query and for its nearest query neighbours."""\n    q, c, nq = s["q"].copy(), s["c"].copy(), len(s["q"])\n    codes = pd.factorize(c.ident)[0].reshape(nq, K)\n    q["r_pn"], q["r_pb"], q["r_rawsum"], q["r_rawmax"] = r["pn"], r["pb"], r["raw"].sum(1), r["raw"].max(1)\n    c["r_raw"], c["r_pc"] = r["raw"].ravel(), r["pc"].ravel()\n    for kind, weight, scale in (("nb", "nw", 1.0), ("sb", "sw", 30.0)):     # embedding / SIFT neighbours\n        evidence, support, pb, w, top = np.zeros((nq, K)), np.zeros((nq, K)), [], [], []\n        for k in range(5):\n            idx, sim = q[f"{kind}{k}"].to_numpy(), q[f"{weight}{k}"].to_numpy() / scale\n            wk = np.clip(sim - 0.4, 0, None) if kind == "nb" else np.clip(sim, 0, 1)\n            agree = ((codes[idx][:, None, :] == codes[:, :, None]) * r["raw"][idx][:, None, :]).sum(2)\n            evidence = np.maximum(evidence, np.minimum(agree, np.clip(sim, 0, 1)[:, None]))\n            support += wk[:, None] * agree\n            pb.append(r["pb"][idx]), w.append(wk), top.append(r["raw"][idx].max(1))\n        pb, w = np.stack(pb, 1), np.stack(w, 1)\n        q[f"{kind}_pb0"], q[f"{kind}_pbw"] = pb[:, 0], (pb * w).sum(1) / np.maximum(w.sum(1), 1e-6)\n        q[f"{kind}_pbmin"], q[f"{kind}_w"], q[f"{kind}_rm0"] = np.where(w > 0, pb, 1).min(1), w.sum(1), top[0]\n        c[f"r_{kind}ev"], c[f"r_{kind}sum"] = evidence.ravel(), support.ravel()\n    return dict(s, q=q, c=c)\n',encoding="utf-8")
(SOURCE / 'README.md').write_text("# Model 7: Nuisance Evidence Retrieval (NER)\n\nMeasured outcome: local open-set MAP@5 rises from 0.934272 to 0.935721, but one of three episodes worsens and the estimated improvement interval includes zero. The submission gate failed. No CSV was generated. See `RESULTS.md` and `results/` for the saved comparison. These are validation scores, not Kaggle scores.\n\nThis experiment tests whether a frozen nuisance projection improves Dalvin Model 3d's complete open-set pipeline. It replaces the raw similarity evidence with similarities between transformed vectors. It preserves the original PCA/LDA retrieval, candidate list, mutual RootSIFT/RANSAC verification, unknown detector, candidate scorers and three prediction-stacking stages. Both versions are retrained for this paired comparison.\n\n## Transformation\n\nFor every fitting identity with multiple photos, subtract its mean descriptor from each photo. Scale its residual rows by `1/sqrt(n-1)` so each identity contributes equal covariance weight. Estimate 64 residual singular directions and retain the leading 16. Reduce each photo's projection along those 16 directions by half, preserve its other coordinates, and normalize the result. Fit one transformation per species. No image features are recomputed from pixels.\n\nThe dimensions (16) and strength (0.5) were frozen in the earlier matching research. The open-set comparison does not tune them or the baseline models using validation outcomes.\n\n## Protocol\n\nSplit identities separately within each species, seed 2052: 20% final validation, 20% metric fitting, and three 20% ranking-training stages. Metric-fitting identities train only the nuisance transformation. Each stacking stage sees its own identity set; earlier models supply predictions to the next stage. Final validation labels are unavailable to every supervised fitting step.\n\nEach stage uses 12 simulated open-set gallery/query episodes. Dalvin's three LightGBM components are fitted per stage: unknown detector, known-query candidate scorer and all-query candidate scorer. Candidate scorers use 400 trees; the original detector uses 300. Both variants share episode assignments, 15 candidate identities, 150 same-species SIFT neighbours and all random seeds. RootSIFT uses a mutual 0.85 ratio test and partial-affine RANSAC with 15-pixel tolerance.\n\nThree predeclared episodes on validation identities contain known queries and entirely held-out unknown identities. Near-duplicate groups stay wholly in gallery or query. The projection affects only raw similarity features, not SIFT neighbour selection, duplicate grouping or original PCA/LDA features. The baseline and transformed versions must have identical candidate lists; runtime assertions enforce this. PCA sees unlabeled episode query features as in Dalvin's implementation; LDA sees gallery labels only.\n\nThis differs from the teammate's original training protocol, which trained stacking with out-of-fold simulated splits over all training identities. Our stricter outer identity separation produces smaller training galleries. It is a controlled comparison of the complete components, not a reproduction of the team's Kaggle score. Multiple validation episodes reuse identities and can reuse query images; their scores are correlated.\n\n## Decision gate\n\nGenerate a submission only when the transformed version has positive pooled and per-episode MAP@5 differences, and pooled known-query and unknown-query MAP@5 do not fall. This gate is declared in the saved configuration before fitting. If it fails, the runner saves the negative result and skips submission generation. Passing does not guarantee a leaderboard improvement.\n\n## Run\n\nCPU is used throughout. From this directory:\n\n```powershell\npython -m pip install -r requirements.txt\npython paired_eval.py --predict-test\n```\n\nData defaults to `~/Downloads/reid_student_data_main`. SIFT archives or extracted NPY directories must be in the same directory or a parent, as supplied in Downloads. `--data` and `--output` override paths. Outputs default to `local_runs/paired_2052`, which is ignored by the parent `.gitignore`. First-run mutual SIFT verification can take substantial time; pairs are cached under `sift_mutual_*.npz` and shared by the baseline and transformed version. Older non-mutual Model 5 caches cannot be reused.\n\n`NER_Local.ipynb` and `NER_Kaggle.ipynb` contain source snapshots and run the same command using the notebook kernel. The Kaggle notebook finds the competition metadata directory under `/kaggle/input`. It requires the competition SIFT files too. Neither notebook submits to Kaggle.\n\n`identity_roles.csv`, `episode_splits.csv`, `config.json`, `paired_models.joblib`, `nuisance_directions.joblib`, `validation_predictions.csv` and `metrics.json` make each run auditable. `submission_nuisance.csv` appears only if the gate passes and `--predict-test` is supplied. The submission then uses the already-frozen stage models and projection with the full labeled gallery, so gallery-size shift remains a limitation.\n\n## Attribution and checks\n\n`dalvin_core.py` derives from `Models/Dalvin/reid_pipeline.py` (normalized source-text SHA256 `65fac657ffb6a63c687d27fbf8938b7bf67dac3d874c2d7569c9597f48ea9f6d`). Besides selecting the nuisance raw representation, the PCA seed is fixed and its dimensionality is bounded by feature count. Data/protocol helpers derive from the earlier Abhay Model 5 runner. Model 5's learned score fusion is not used.\n\n`python -m unittest test_integration.py` checks exact baseline feature parity against the original Dalvin feature function and confirms that labels outside the metric-fitting set do not affect the learned transformation. The original feature function is retained in `baseline_species_features.py` for this check.\n\n`python audit_results.py --run local_runs/paired_2052` independently recomputes scores, validates identity roles and prediction labels, and estimates uncertainty by resampling animal identities. Both notebooks run this output audit after evaluation.\n",encoding="utf-8")
(SOURCE / 'requirements.txt').write_text('numpy>=1.26,<2.5\npandas>=2.1,<3\nscipy>=1.11,<2\nscikit-learn>=1.4,<2\nlightgbm>=4,<5\nopencv-python>=4.8,<5\njoblib>=1.3,<2\nthreadpoolctl>=3,<4\n',encoding="utf-8")
(SOURCE / 'test_integration.py').write_text('"""Verify baseline feature parity and supervised identity separation."""\nimport ast\nimport unittest\nimport tempfile\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom scipy import sparse\n\nimport dalvin_core as b\nimport paired_eval\nimport data_protocol\n\n\nclass IntegrationTest(unittest.TestCase):\n    def setUp(self):\n        rng = np.random.default_rng(5)\n        b.X = b.unit(rng.normal(size=(100, 24)).astype(np.float32))\n        b.NX = b.X\n        b.Y = np.array([f\'id{i // 4}\' for i in range(100)])\n        b.SP = np.array([\'hyena\'] * 100, dtype=object)\n        b.VIEW = np.arange(100) % 10\n        b.INL = sparse.csr_matrix((100, 100))\n        b.GOOD = b.INL.copy()\n        b.K = 15\n\n    def test_baseline_species_features_match_original(self):\n        reference = Path(__file__).with_name(\'baseline_species_features.py\')\n        namespace = dict(vars(b))\n        exec(compile(reference.read_text(), str(reference), \'exec\'), namespace)\n        g = np.r_[np.arange(0, 100, 4), np.arange(1, 100, 4)]\n        q = np.arange(2, 100, 4)\n        actual = b.species_features(g, q, \'hyena\')\n        expected = namespace[\'species_features\'](g, q, \'hyena\')\n        for a, e in zip(actual, expected):\n            pd.testing.assert_frame_equal(a, e)\n\n    def test_nuisance_fit_never_reads_other_identity_labels(self):\n        b.SP[50:] = \'leopard\'\n        fit = np.r_[np.arange(0, 40), np.arange(60, 100)]\n        first, directions = paired_eval.nuisance_features(fit, seed=11)\n        b.Y[40:60] = \'unused-label\'\n        second, again = paired_eval.nuisance_features(fit, seed=11)\n        np.testing.assert_allclose(first, second)\n        for species in directions:\n            np.testing.assert_allclose(directions[species], again[species])\n        np.testing.assert_allclose(np.linalg.norm(first, axis=1), 1, atol=1e-6)\n\n    def test_sift_checkpoint_preserves_completed_pairs(self):\n        with tempfile.TemporaryDirectory() as temporary:\n            cache = object.__new__(data_protocol.SiftCache)\n            cache.path = Path(temporary) / \'pairs.npz\'\n            cache.done = {(1, 3): (6, 4), (2, 5): (12, 10)}\n            cache.save()\n            with np.load(cache.path) as saved:\n                recovered = {tuple(p): tuple(v) for p, v in zip(saved[\'pairs\'], saved[\'values\'])}\n            self.assertEqual(recovered, cache.done)\n            self.assertFalse(cache.path.with_suffix(\'.tmp.npz\').exists())\n\n\nif __name__ == \'__main__\':\n    unittest.main()\n',encoding="utf-8")
(SOURCE / 'baseline_species_features.py').write_text('"""Unmodified Dalvin 3d feature function, retained for parity verification."""\ndef species_features(gi, qi, sp):\n    """Query frame and (query, candidate) frame for one species; gi/qi are global image indices."""\n    Rg, Rq, yg, vg, vq = X[gi], X[qi], Y[gi], VIEW[gi], VIEW[qi]\n    Lg, Lq = embed(Rg, yg, Rq)\n    ids, inv = np.unique(yg, return_inverse=True)\n    n, nq = len(ids), len(qi)\n    sim_l, sim_r = Lq @ Lg.T, Rq @ Rg.T\n    lda1, lda2 = per_identity(sim_l, inv, n)\n    raw1, raw2 = per_identity(sim_r, inv, n)\n    centroid = unit(np.stack([Lg[inv == j].mean(0) for j in range(n)]))\n    cen = Lq @ centroid.T\n    inl_g, good_g = INL[qi][:, gi].toarray().astype(np.float32), GOOD[qi][:, gi].toarray().astype(np.float32)\n    sift1, sift2 = per_identity(inl_g, inv, n)\n    good1, _ = per_identity(good_g, inv, n)\n    support = (inl_g >= 7).astype(np.float32) @ np.eye(n, dtype=np.float32)[inv]     # gallery images confirmed per identity\n    same_view = np.full((nq, n), np.nan)\n    for j in range(n):\n        for v in np.unique(vq):\n            cols = np.where((inv == j) & (vg == v))[0]\n            if len(cols):\n                same_view[vq == v, j] = sim_l[np.ix_(vq == v, cols)].max(1)\n    # viewpoint coverage: flanks of one animal look different, so unseen sides cannot be matched\n    by_view = np.stack([np.bincount(inv[vg == v], minlength=n) for v in range(len(VIEWS))], 1)\n    by_flank = np.stack([by_view[:, FLANK == f].sum(1) for f in range(3)], 1)\n    seen = vq >= 0\n    n_view = np.where(seen[:, None], by_view[:, vq.clip(0)].T, np.nan)\n    n_flank = np.where(seen[:, None], by_flank[:, FLANK[vq.clip(0)]].T, np.nan)\n    # transductive: neighbours among the queries themselves\n    qq_r, qq_l, qq_s = Rq @ Rq.T, Lq @ Lq.T, INL[qi][:, qi].toarray().astype(np.float32)\n    np.fill_diagonal(qq_r, -1), np.fill_diagonal(qq_l, -1)\n    nb, sb = np.argsort(-qq_l, 1)[:, :5], np.argsort(-qq_s, 1)[:, :5]\n    nbs, sbs = np.take_along_axis(qq_l, nb, 1), np.take_along_axis(qq_s, sb, 1)\n    t_l = np.take_along_axis(lda1, np.argsort(-lda1, 1)[:, :3], 1)\n    t_r = -np.sort(-raw1, 1)[:, :3]\n    t_s = -np.sort(-sift1, 1)[:, :3]\n    g_top = -np.sort(-sim_l, 1)\n    q = pd.DataFrame({\n        "sp": int(sp == "hyena"), "view": vq, "ng": len(gi),\n        "l1": t_l[:, 0], "l2": t_l[:, 1], "l3": t_l[:, 2], "r1": t_r[:, 0], "r2": t_r[:, 1], "r3": t_r[:, 2],\n        "lm12": t_l[:, 0] - t_l[:, 1], "rm12": t_r[:, 0] - t_r[:, 1],\n        "gd5": g_top[:, :5].mean(1), "gd20": g_top[:, :20].mean(1),\n        "qq1": nbs[:, 0], "qq3": nbs[:, :3].mean(1), "qq5": nbs.mean(1), "qqr1": qq_r.max(1),\n        "qn6": (qq_r > .6).sum(1), "qn7": (qq_r > .7).sum(1), "qn8": (qq_r > .8).sum(1),\n        "qq_minus_g": nbs[:, 0] - t_l[:, 0], "nb_g1": t_l[nb[:, 0], 0],\n        "s1": t_s[:, 0], "s2": t_s[:, 1], "s3": t_s[:, 2], "sg": good_g.max(1),\n        "qs1": sbs[:, 0], "qs3": sbs[:, :3].mean(1), "qsn10": (qq_s >= 10).sum(1), "qsn20": (qq_s >= 20).sum(1),\n        "qs_minus_s": sbs[:, 0] - t_s[:, 0], "nbs_s1": t_s[sb[:, 0], 0],\n    })\n    for k in range(5):\n        q[f"nb{k}"], q[f"nw{k}"], q[f"sb{k}"], q[f"sw{k}"] = nb[:, k], nbs[:, k], sb[:, k], sbs[:, k]\n    # candidates: best identities by embedding similarity, lightly boosted by SIFT evidence\n    cand = np.argsort(-(lda1 + SIFT_PULL * np.tanh(sift1 / 15)), 1)[:, :K]\n    r, c = np.repeat(np.arange(nq), K), cand.ravel()\n    nb_ev = np.max(np.minimum(nbs[:, :, None], np.stack([lda1[nb[:, k]] for k in range(5)], 1)), 1)\n    sb_ev = np.max(np.minimum(sbs[:, :, None], np.stack([sift1[sb[:, k]] for k in range(5)], 1)), 1)\n    rank_of = lambda m: (-m).argsort(1).argsort(1)[r, c]\n    cd = pd.DataFrame({\n        "q": r, "c": c, "rank": np.tile(np.arange(K), nq), "ident": ids[c],\n        "mxr": raw1[r, c], "m2r": raw2[r, c], "mxl": lda1[r, c], "m2l": lda2[r, c], "cl": cen[r, c],\n        "gapl": lda1[r, c] - t_l[r, 0], "gapr": raw1[r, c] - t_r[r, 0], "same": same_view[r, c],\n        "cnt": np.bincount(inv)[c], "nbev": nb_ev[r, c], "sbev": sb_ev[r, c],\n        "rk_r": rank_of(raw1), "rk_c": rank_of(cen), "rk_s": rank_of(sift1),\n        "n_view": n_view[r, c], "n_flank": n_flank[r, c], "support": support[r, c], "sx": sift1[r, c], "s2x": sift2[r, c], "gx": good1[r, c], "sgap": sift1[r, c] - t_s[r, 0],\n    })\n    return q, cd\n',encoding="utf-8")
(SOURCE / 'audit_results.py').write_text('"""Recompute paired scores and check identity roles and episode assignments."""\nimport argparse\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\n\ndef main():\n    ap = argparse.ArgumentParser(description=__doc__)\n    ap.add_argument(\'--run\', type=Path, default=Path(\'local_runs/paired_2052\'))\n    args = ap.parse_args()\n    root = args.run\n    config = json.loads((root / \'config.json\').read_text())\n    report = json.loads((root / \'metrics.json\').read_text())\n    train = pd.read_csv(Path(config[\'data\']) / \'train.csv\').set_index(\'id\')\n    roles = pd.read_csv(root / \'identity_roles.csv\').set_index(\'id\')\n    episodes = pd.read_csv(root / \'episode_splits.csv\')\n    rows = pd.read_csv(root / \'validation_predictions.csv\')\n    assert roles.index.is_unique and set(roles.index) == set(train.index)\n    assert roles.groupby(\'identity\').role.nunique().eq(1).all()\n    assert not episodes.duplicated([\'episode\', \'id\']).any()\n    galleries = {}\n    for seed, part in episodes.groupby(\'episode\'):\n        assert set(part.role) == set(roles.loc[part.id, \'role\'])\n        gallery = set(train.loc[part.loc[part.assignment == \'gallery\', \'id\'], \'identity\'])\n        known = set(train.loc[part.loc[part.assignment == \'known_query\', \'id\'], \'identity\'])\n        unknown = set(train.loc[part.loc[part.assignment == \'unknown_query\', \'id\'], \'identity\'])\n        assert known <= gallery and not unknown & gallery\n        galleries[int(seed)] = gallery\n    assert set(roles.loc[rows.id, \'role\']) == {\'validation\'}\n    assert set(rows.episode) == set(config[\'evaluation_episodes\'])\n    assert rows.unknown.eq(rows.truth == \'new_individual\').all()\n    assignments = episodes.set_index([\'episode\', \'id\']).assignment\n    species_catalog = train.groupby(\'species\').identity.apply(set).to_dict()\n    for row in rows.itertuples():\n        assert row.identity == train.loc[row.id, \'identity\']\n        assert row.species == train.loc[row.id, \'species\']\n        assert row.truth == (\'new_individual\' if row.unknown else row.identity)\n        assert assignments.loc[(row.episode, row.id)] == (\'unknown_query\' if row.unknown else \'known_query\')\n        for method in (\'baseline\', \'nuisance\'):\n            labels = getattr(row, method + \'_predictions\').split()\n            assert len(labels) == len(set(labels)) == 5\n            assert set(labels) <= galleries[row.episode] | {\'new_individual\'}\n            assert set(labels) <= species_catalog[row.species] | {\'new_individual\'}\n            rr = next((1 / (j + 1) for j, label in enumerate(labels) if label == row.truth), 0.)\n            assert np.isclose(rr, getattr(row, method + \'_rr5\'))\n    for method in (\'baseline\', \'nuisance\'):\n        assert np.isclose(report[method][\'map5\'], rows[method + \'_rr5\'].mean())\n    rows[\'delta\'] = rows.nuisance_rr5 - rows.baseline_rr5\n    # Resample complete animal identities, preserving repeated query appearances.\n    groups = rows.groupby([\'species\', \'identity\']).delta.agg([\'sum\', \'size\'])\n    rng = np.random.default_rng(7052)\n    bootstrap = []\n    for _ in range(2000):\n        numerator, denominator = 0., 0\n        for _, part in groups.groupby(level=\'species\'):\n            sample = part.iloc[rng.integers(len(part), size=len(part))]\n            numerator += sample[\'sum\'].sum()\n            denominator += sample[\'size\'].sum()\n        bootstrap.append(numerator / denominator)\n    audit = {\'identity_role_and_score_checks\': \'passed\', \'query_appearances\': len(rows),\n             \'unique_query_images\': int(rows.id.nunique()), \'queried_identities\': len(groups),\n             \'pooled_delta\': float(rows.delta.mean()),\n             \'identity_cluster_bootstrap_95_interval\': np.quantile(bootstrap, [.025, .975]).tolist(),\n             \'note\': \'Conditional on this fitted model pair and outer identity partition; not a leaderboard confidence interval.\'}\n    (root / \'audit.json\').write_text(json.dumps(audit, indent=2))\n    print(json.dumps(audit, indent=2))\n\n\nif __name__ == \'__main__\':\n    main()\n',encoding="utf-8")
print("Source snapshot:",SOURCE.resolve())


## Data and fixed configuration
Data defaults to the Downloads folder. SIFT directories remain alongside the metadata directory.

In [ ]:
from importlib.util import find_spec
missing=[m for m in ('numpy','pandas','scipy','sklearn','lightgbm','cv2','joblib','threadpoolctl') if find_spec(m) is None]
if missing:
    raise ImportError(f"Install requirements in the active kernel; missing: {missing}")
DATA=Path.home() / "Downloads" / "reid_student_data_main"
OUTPUT=RUN_DIR / "results"
SEED=2052
SIMS=12
WORKERS=4
NEIGHBOURS=150
TREES=400
PREDICT_TEST=True
print("Data:",DATA)
print("Output:",OUTPUT.resolve())


## Train both versions and evaluate
Both versions use identical identity roles, simulations, candidates and SIFT matches. Only fit-role labels estimate the transformation; validation labels are reserved for final scoring.

In [ ]:
command=[sys.executable,str((SOURCE/'paired_eval.py').resolve()),'--data',str(DATA.resolve()),
         '--output',str(OUTPUT.resolve()),'--seed',str(SEED),'--sims',str(SIMS),
         '--workers',str(WORKERS),'--neighbours',str(NEIGHBOURS),'--trees',str(TREES)]
if PREDICT_TEST: command.append('--predict-test')
print(' '.join(command),flush=True)
subprocess.run(command,check=True)


## Paired result
The gate requires a positive pooled and per-episode difference without lower pooled known or unknown MAP@5. Passing this test does not guarantee a leaderboard gain.

In [ ]:
subprocess.run([sys.executable,str((SOURCE/'audit_results.py').resolve()),'--run',str(OUTPUT.resolve())],check=True)
import pandas as pd
report=json.loads((OUTPUT/'metrics.json').read_text())
display(pd.DataFrame({m:report[m] for m in ('baseline','nuisance')}).T)
print('Pooled MAP@5 difference:',report['delta'])
print('Submission gate passed:',report['gate_passed'])
for episode, scores in report['episodes'].items():
    print('Episode',episode,'MAP@5 difference:',scores['delta'])
submission=OUTPUT/'submission_nuisance.csv'
if submission.exists() and report['gate_passed']:
    print('Submission CSV:',submission.resolve())
else:
    print('No submission promoted; review paired validation results.')
